# Brasil em Dados

## Preparação e integração das bases

Este notebook prepara os dados educacionais do INEP e os dados
socioeconômicos do IBGE para a análise dos municípios brasileiros.

A base educacional utiliza indicadores de 2023 dos anos finais do
Ensino Fundamental. A base socioeconômica utiliza informações do
Censo Demográfico de 2022.

As bases serão integradas pelo código do município. Os valores ausentes
serão identificados e mantidos nesta etapa; o tratamento específico
desses valores será definido nos notebooks de análise.

## 1. Configuração do ambiente

Nesta etapa serão importadas as bibliotecas necessárias e definidos os
caminhos para os dados brutos e processados.

Este exemplo considera que o notebook está dentro de uma pasta chamada
`notebooks`. Se ele ficar na pasta principal do projeto, troque
`Path("..")` por `Path(".")`.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

INEP_DIR = DATA_RAW / "inep"
IBGE_DIR = DATA_RAW / "ibge"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Pasta do projeto:", PROJECT_ROOT.resolve())
print("Pasta dos dados brutos:", DATA_RAW.resolve())
print("Pasta dos dados processados:", DATA_PROCESSED.resolve())

Pasta do projeto: C:\Users\manel\Documents\Ciencia de Dados\brasil_em_dados
Pasta dos dados brutos: C:\Users\manel\Documents\Ciencia de Dados\brasil_em_dados\data\raw
Pasta dos dados processados: C:\Users\manel\Documents\Ciencia de Dados\brasil_em_dados\data\processed


## 2. Carregamento dos dados educacionais — INEP

Será utilizada a planilha de resultados municipais dos anos finais do
Ensino Fundamental. A análise considera os indicadores de 2023 da rede
pública.

In [3]:
arquivo_ideb = (
    INEP_DIR
    / "divulgacao_anos_finais_municipios_2025.xlsx"
)

df_ideb = pd.read_excel(
    arquivo_ideb,
    sheet_name="IDEB_AF_MUNICÍPIOS",
    header=9
)

print("Dimensões da base original:", df_ideb.shape)
print("Colunas disponíveis:")
display(df_ideb.columns.to_frame(index=False, name="coluna"))

display(df_ideb.head())

Dimensões da base original: (14428, 122)
Colunas disponíveis:


,coluna
0,SG_UF
1,CO_MUNICIPIO
2,NO_MUNICIPIO
3,REDE
4,VL_APROVACAO_2005_SI_4
...,...
117,VL_PROJECAO_2013
118,VL_PROJECAO_2015
119,VL_PROJECAO_2017
120,VL_PROJECAO_2019


,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,REDE,VL_APROVACAO_2005_SI_4,VL_APROVACAO_2005_1,VL_APROVACAO_2005_2,VL_APROVACAO_2005_3,VL_APROVACAO_2005_4,VL_INDICADOR_REND_2005,...,VL_OBSERVADO_2023,VL_OBSERVADO_2025,VL_PROJECAO_2007,VL_PROJECAO_2009,VL_PROJECAO_2011,VL_PROJECAO_2013,VL_PROJECAO_2015,VL_PROJECAO_2017,VL_PROJECAO_2019,VL_PROJECAO_2021
0,RO,1100015.0,Alta Floresta D'Oeste,Estadual,75.9,69.6,75.9,83.2,77,0.76119,...,4.9,5.1,3.5,3.7,4,4.4,4.8,5,5.3,5.5
1,RO,1100015.0,Alta Floresta D'Oeste,Municipal,77.6,68.1,74.3,83.3,89,0.778453,...,4,-,3.3,3.4,3.7,4.1,4.5,4.7,5,5.3
2,RO,1100015.0,Alta Floresta D'Oeste,Pública,76.7,68.8,75,83.2,82.2,0.768449,...,4.7,5,3.5,3.7,4,4.4,4.7,5,5.3,5.5
3,RO,1100023.0,Ariquemes,Estadual,79.2,84.2,79.9,76.7,75.7,0.78989,...,4.8,4.9,3.7,3.8,4.1,4.5,4.9,5.1,5.4,5.6
4,RO,1100023.0,Ariquemes,Municipal,77.6,72.2,76.9,81.1,86.3,0.787832,...,4.4,4.5,3.3,3.5,3.7,4.1,4.5,4.8,5,5.3


## 3. Seleção e pré-processamento dos dados educacionais

Como a unidade de análise é o município, serão mantidos o código e o nome
do município, a unidade federativa e os indicadores selecionados para
2023. O recorte será feito para a rede pública.

Os nomes das colunas serão simplificados para facilitar as análises
posteriores.

In [4]:
colunas_educacao = [
    "SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "REDE",
    "VL_APROVACAO_2023_SI_4",
    "VL_INDICADOR_REND_2023",
    "VL_NOTA_MATEMATICA_2023",
    "VL_NOTA_PORTUGUES_2023",
    "VL_NOTA_MEDIA_2023",
    "VL_OBSERVADO_2023"
]

colunas_ausentes = [
    coluna
    for coluna in colunas_educacao
    if coluna not in df_ideb.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas esperadas não encontradas na planilha: {colunas_ausentes}"
    )

df_educacao = (
    df_ideb.loc[
        df_ideb["REDE"].astype("string").str.strip().eq("Pública"),
        colunas_educacao
    ]
    .copy()
)

df_educacao = df_educacao.rename(
    columns={
        "SG_UF": "uf",
        "CO_MUNICIPIO": "codigo_municipio",
        "NO_MUNICIPIO": "municipio",
        "VL_APROVACAO_2023_SI_4": "taxa_aprovacao_2023",
        "VL_INDICADOR_REND_2023": "indicador_rendimento_2023",
        "VL_NOTA_MATEMATICA_2023": "nota_matematica_2023",
        "VL_NOTA_PORTUGUES_2023": "nota_portugues_2023",
        "VL_NOTA_MEDIA_2023": "nota_media_2023",
        "VL_OBSERVADO_2023": "ideb_2023"
    }
)

display(df_educacao.head())
print("Registros da rede pública:", len(df_educacao))

,uf,codigo_municipio,municipio,REDE,taxa_aprovacao_2023,indicador_rendimento_2023,nota_matematica_2023,nota_portugues_2023,nota_media_2023,ideb_2023
2,RO,1100015.0,Alta Floresta D'Oeste,Pública,93.2,0.931733,254.11,246.21,5.005333,4.7
5,RO,1100023.0,Ariquemes,Pública,93.8,0.937832,251.95,251.99,5.065667,4.8
7,RO,1100031.0,Cabixi,Pública,99.7,0.997227,247.93,252.78,5.011833,5
10,RO,1100049.0,Cacoal,Pública,98.8,0.988228,247.61,249.67,4.954667,4.9
13,RO,1100056.0,Cerejeiras,Pública,96.7,0.966651,255.25,252.94,5.1365,5


Registros da rede pública: 5569


In [5]:
colunas_numericas_inep = [
    "taxa_aprovacao_2023",
    "indicador_rendimento_2023",
    "nota_matematica_2023",
    "nota_portugues_2023",
    "nota_media_2023",
    "ideb_2023"
]

for coluna in colunas_numericas_inep:
    df_educacao[coluna] = pd.to_numeric(
        df_educacao[coluna],
        errors="coerce"
    )

df_educacao["codigo_municipio"] = (
    pd.to_numeric(
        df_educacao["codigo_municipio"],
        errors="coerce"
    )
    .astype("Int64")
)

df_educacao["uf"] = (
    df_educacao["uf"]
    .astype("string")
    .str.strip()
)

df_educacao["municipio"] = (
    df_educacao["municipio"]
    .astype("string")
    .str.strip()
)

print("Municípios únicos:", df_educacao["codigo_municipio"].nunique())
print("Códigos ausentes:", df_educacao["codigo_municipio"].isna().sum())
print(
    "Códigos duplicados:",
    df_educacao["codigo_municipio"].duplicated().sum()
)

display(
    df_educacao.isna()
    .sum()
    .to_frame("valores_ausentes")
)

Municípios únicos: 5569
Códigos ausentes: 0
Códigos duplicados: 0


,valores_ausentes
uf,0
codigo_municipio,0
municipio,0
REDE,0
taxa_aprovacao_2023,49
indicador_rendimento_2023,49
nota_matematica_2023,186
nota_portugues_2023,186
nota_media_2023,186
ideb_2023,187


## 4. Carregamento e pré-processamento dos dados socioeconômicos — IBGE

Serão utilizadas as variáveis de população residente e rendimento
domiciliar mensal per capita, referentes ao Censo Demográfico de 2022.

O código do município será mantido como identificador para a integração
com a base educacional.

In [6]:
arquivo_ibge = (
    IBGE_DIR
    / "ibge_municipios_censo_2022.csv"
)

df_ibge = pd.read_csv(
    arquivo_ibge,
    encoding="utf-8-sig"
)

colunas_ibge_esperadas = [
    "codigo_municipio",
    "uf",
    "municipio",
    "populacao_2022",
    "renda_domiciliar_per_capita_2022",
    "observacao"
]

colunas_ausentes = [
    coluna
    for coluna in colunas_ibge_esperadas
    if coluna not in df_ibge.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas esperadas não encontradas no CSV do IBGE: {colunas_ausentes}"
    )

df_ibge = df_ibge[colunas_ibge_esperadas].copy()

df_ibge["codigo_municipio"] = (
    pd.to_numeric(
        df_ibge["codigo_municipio"],
        errors="coerce"
    )
    .astype("Int64")
)

df_ibge["populacao_2022"] = pd.to_numeric(
    df_ibge["populacao_2022"],
    errors="coerce"
).astype("Int64")

df_ibge["renda_domiciliar_per_capita_2022"] = pd.to_numeric(
    df_ibge["renda_domiciliar_per_capita_2022"],
    errors="coerce"
)

df_ibge["uf"] = df_ibge["uf"].astype("string").str.strip()
df_ibge["municipio"] = df_ibge["municipio"].astype("string").str.strip()
df_ibge["observacao"] = df_ibge["observacao"].astype("string")

print("Dimensões da base do IBGE:", df_ibge.shape)
print("Municípios únicos:", df_ibge["codigo_municipio"].nunique())
print(
    "Códigos duplicados:",
    df_ibge["codigo_municipio"].duplicated().sum()
)

display(df_ibge.head())
display(
    df_ibge.isna()
    .sum()
    .to_frame("valores_ausentes")
)

Dimensões da base do IBGE: (5571, 6)
Municípios únicos: 5571
Códigos duplicados: 0


,codigo_municipio,uf,municipio,populacao_2022,renda_domiciliar_per_capita_2022,observacao
0,1200013,AC,Acrelândia,14021,880.50,<NA>
1,1200054,AC,Assis Brasil,8100,775.86,<NA>
2,1200104,AC,Brasiléia,26000,914.18,<NA>
3,1200138,AC,Bujari,12917,709.71,<NA>
4,1200179,AC,Capixaba,10392,806.69,<NA>


,valores_ausentes
codigo_municipio,0
uf,0
municipio,0
populacao_2022,1
renda_domiciliar_per_capita_2022,1
observacao,5570


## 5. Integração das bases

As bases do INEP e do IBGE serão integradas pelo código do município.
Antes da integração, será verificado se há municípios presentes em apenas
uma das bases.

A base integrada terá como referência os municípios presentes no INEP.
Assim, os municípios sem correspondência no IBGE continuarão nela, com
valores ausentes nas variáveis socioeconômicas.

In [7]:
comparacao_municipios = (
    df_educacao[["codigo_municipio"]]
    .merge(
        df_ibge[["codigo_municipio"]],
        on="codigo_municipio",
        how="outer",
        indicator=True
    )
)

display(
    comparacao_municipios["_merge"]
    .value_counts()
    .to_frame("quantidade")
)

municipios_so_inep = comparacao_municipios.loc[
    comparacao_municipios["_merge"].eq("left_only"),
    "codigo_municipio"
]

municipios_so_ibge = comparacao_municipios.loc[
    comparacao_municipios["_merge"].eq("right_only"),
    "codigo_municipio"
]

print("Códigos presentes apenas no INEP:", len(municipios_so_inep))
print("Códigos presentes apenas no IBGE:", len(municipios_so_ibge))

,quantidade
_merge,
both,5569
right_only,2
left_only,0


Códigos presentes apenas no INEP: 0
Códigos presentes apenas no IBGE: 2


In [8]:
colunas_ibge_integracao = [
    "codigo_municipio",
    "populacao_2022",
    "renda_domiciliar_per_capita_2022",
    "observacao"
]

df_integrado = df_educacao.merge(
    df_ibge[colunas_ibge_integracao],
    on="codigo_municipio",
    how="left",
    validate="one_to_one"
)

print("Registros do INEP:", len(df_educacao))
print("Registros após integração:", len(df_integrado))
print("Municípios únicos:", df_integrado["codigo_municipio"].nunique())
print(
    "Códigos duplicados após integração:",
    df_integrado["codigo_municipio"].duplicated().sum()
)

display(
    df_integrado[
        [
            "populacao_2022",
            "renda_domiciliar_per_capita_2022",
            "ideb_2023"
        ]
    ]
    .isna()
    .sum()
    .to_frame("valores_ausentes")
)

display(df_integrado.head())

Registros do INEP: 5569
Registros após integração: 5569
Municípios únicos: 5569
Códigos duplicados após integração: 0


,valores_ausentes
populacao_2022,1
renda_domiciliar_per_capita_2022,1
ideb_2023,187


,uf,codigo_municipio,municipio,REDE,taxa_aprovacao_2023,indicador_rendimento_2023,nota_matematica_2023,nota_portugues_2023,nota_media_2023,ideb_2023,populacao_2022,renda_domiciliar_per_capita_2022,observacao
0,RO,1100015,Alta Floresta D'Oeste,Pública,93.2,0.931733,254.11,246.21,5.005333,4.7,21494,1210.60,<NA>
1,RO,1100023,Ariquemes,Pública,93.8,0.937832,251.95,251.99,5.065667,4.8,96833,1458.57,<NA>
2,RO,1100031,Cabixi,Pública,99.7,0.997227,247.93,252.78,5.011833,5.0,5351,1323.79,<NA>
3,RO,1100049,Cacoal,Pública,98.8,0.988228,247.61,249.67,4.954667,4.9,86887,1625.88,<NA>
4,RO,1100056,Cerejeiras,Pública,96.7,0.966651,255.25,252.94,5.136500,5.0,15890,1554.33,<NA>


## 6. Salvamento das bases processadas

A base integrada será salva em `data/processed`. Os valores ausentes serão
preservados neste arquivo; a escolha de como tratá-los será feita nas
análises seguintes, conforme as variáveis utilizadas.

In [9]:
arquivo_educacao_processado = (
    DATA_PROCESSED / "educacao_municipios_2023.csv"
)

arquivo_base_integrada = (
    DATA_PROCESSED / "brasil_em_dados_municipios.csv"
)

df_educacao.to_csv(
    arquivo_educacao_processado,
    index=False,
    encoding="utf-8-sig"
)

df_integrado.to_csv(
    arquivo_base_integrada,
    index=False,
    encoding="utf-8-sig"
)

print("Base educacional salva em:", arquivo_educacao_processado.resolve())
print("Base integrada salva em:", arquivo_base_integrada.resolve())

Base educacional salva em: C:\Users\manel\Documents\Ciencia de Dados\brasil_em_dados\data\processed\educacao_municipios_2023.csv
Base integrada salva em: C:\Users\manel\Documents\Ciencia de Dados\brasil_em_dados\data\processed\brasil_em_dados_municipios.csv
